# Himawari-9 over Anak Krakatau - 3. Frames to animation

Benny Istanto - https://benny.istan.to

---

MP4 is the primary output. Roughly 400 frames at 5000 px wide is a few tens of
megabytes as H.264; the same thing as a GIF would be gigabytes, dithered down
to 256 colours. GIF is still produced, narrower, for the places that insist on
one.

**Gaps are held, not dropped.** Where a timeslot has no frame - AHI's 02:40
and 14:40 UTC housekeeping, or night in a daylight-only composite - the
previous frame repeats. Playback then runs at a constant amount of real time
per second of video. Dropping the gap instead would make the plume appear to
jump, which reads as a data artefact rather than what it is.

## 1. Configuration and inventory

In [ ]:
import sys, warnings
from pathlib import Path

# The package lives in src/ and is not installed, so put it on the path.
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

from himawari import project

cfg = project.load(ROOT / "config.yml")
cfg.make_dirs()
print(cfg.summary())

from himawari import render as R
from himawari import animate as A

R.use_local_config()
anim = cfg.animation
fps = anim.get("fps", 12)

inventory = {}
for p in cfg.products:
    frames = R.rendered_frames(cfg.frames_dir, p.composite, p.grid)
    inventory[p.composite] = (p, frames)
    print(f"{p.composite:20s} {A.describe(frames, fps)}")

## 2. MP4

`fill_gaps` bridges missing timeslots so playback keeps a constant amount of
real time per second. Short gaps are blended between the frames either side,
which reads as motion; longer ones repeat the previous frame, which reads as a
pause and is the honest depiction of data that is simply absent. `max_blend`
is the cutoff, and `daylight_test` stops a gap that straddles sunrise from
dissolving true colour into an infrared RGB. Nothing synthetic is written to
disk.

`interpolate="minterpolate"` goes further, synthesising a frame between every
pair so the native ten-minute cadence plays as an effective five minutes. It
runs in the encoder and does not change the duration, so whatever
real-time-per-second rate `fps` sets stays true. On a hold-out test over the
onset it reconstructed withheld real frames 17.8% better (RMSE, moving pixels)
than averaging the two neighbours, which smears the dark core at the vent into
two half-bright copies. It costs roughly three and a half minutes per encode
here, and the frames it adds are synthesised, not observed.

One caveat. It suits a continuous sequence. A daylight-only composite such as
`true_color` has twelve-hour night gaps, and motion-compensating across one
morphs between two unrelated scenes rather than cutting between them.

In [ ]:
# Interpolation is off by default here because this cell loops over every
# product: about three and a half minutes per encode adds up. The two
# published animations are built with it on. See the note above for when it
# is a bad idea.
SMOOTH = False

made = {}
for name, (p, frames) in inventory.items():
    if not frames:
        print(f"{name}: no frames, skipping")
        continue
    seq = A.fill_gaps(frames, step_minutes=cfg.step_minutes, mode="blend",
                      max_blend=3, max_hold=6,
                      daylight_test=lambda t: R.is_daylight(t, p.grid))
    out = cfg.output_dir / f"krakatau_{name}_{p.grid.name}.mp4"
    A.to_mp4(seq, out, fps=fps, max_width=anim.get("mp4_max_width"),
             interpolate="minterpolate" if SMOOTH else None,
             out_fps=fps * 2 if SMOOTH else None)
    made[name] = out
    print(f"{out.name:48s} {len(seq):4d} frames  {out.stat().st_size/2**20:6.1f} MB")

### Full-resolution cut

The cell above downscales to `animation.mp4_max_width` for something that
scrubs smoothly in a browser. For the true-colour series it is worth also
keeping a native-resolution encode - that is what the 0.5 km download bought.

In [ ]:
tc = inventory.get("true_color")
if tc and tc[1]:
    p, frames = tc
    seq = A.fill_gaps(frames, step_minutes=cfg.step_minutes, mode="blend",
                      max_blend=3, max_hold=6,
                      daylight_test=lambda t: R.is_daylight(t, p.grid))
    out = cfg.output_dir / f"krakatau_true_color_{p.grid.name}_native.mp4"
    A.to_mp4(seq, out, fps=fps, quality=8, max_width=None)
    print(f"{out.name}  {out.stat().st_size/2**20:.1f} MB  at {p.grid.shape[1]} px wide")

## 3. GIF

Ash and dust survive a 128-colour palette well - they are already close to
false colour. True colour does not; use the MP4 for that.

In [ ]:
for name in ("ash", "dust"):
    entry = inventory.get(name)
    if not entry or not entry[1]:
        continue
    p, frames = entry
    seq = A.fill_gaps(frames, step_minutes=cfg.step_minutes, mode="blend",
                      max_blend=3, max_hold=6,
                      daylight_test=lambda t: R.is_daylight(t, p.grid))
    # Decimate hard. GIF is a poor fit for this material: measured on the ash
    # series, hourly steps at 64 colours still cost 14 MB, against every one of
    # 433 frames in an MP4 for less. Handing the full sequence to to_gif would
    # produce a file of well over 100 MB. Use the MP4 above unless something
    # downstream genuinely cannot take video.
    every = max(1, 60 // cfg.step_minutes)      # one frame per hour
    out = cfg.output_dir / f"krakatau_{name}.gif"
    A.to_gif(seq[::every], out, fps=anim.get("gif_fps", 10),
             max_width=anim.get("gif_max_width", 900))
    print(f"{out.name:36s} {out.stat().st_size/2**20:6.1f} MB")

## 4. Side by side

True colour next to ash RGB makes the argument for the IR product directly:
the left panel goes black at sunset while the right panel keeps tracking the
plume. Panels are matched on observation time rather than list position, so
the two never drift out of sync.

In [ ]:
tc, ash = inventory.get("true_color"), inventory.get("ash")
if tc and ash and tc[1] and ash[1]:
    pairs = A.side_by_side(tc[1], ash[1], cfg.temp_dir / "pairs")
    print(f"{len(pairs)} paired frames (only where both exist)")
    if pairs:
        out = cfg.output_dir / "krakatau_truecolor_vs_ash.mp4"
        A.to_mp4(pairs, out, fps=fps, max_width=1920)
        print(out, f"{out.stat().st_size/2**20:.1f} MB")

## 5. Look at the results

In [ ]:
from IPython.display import Video, display

for name, path in made.items():
    print(name)
    # embed=False deliberately. embed=True base64s the whole file into the
    # notebook: a 60 MB video becomes an 80 MB cell, and the notebook has to
    # be stripped before it can be committed.
    display(Video(str(path.relative_to(ROOT)), embed=True, width=900))

---

## Where this came from

- **Data**: Himawari-9 AHI Level 1b full disk (Himawari Standard Data), JMA,
  distributed by JAXA P-Tree and mirrored by NOAA on AWS Open Data.
- **Processing**: [Satpy](https://satpy.readthedocs.io) `ahi_hsd` reader and
  composites, [Pyresample](https://pyresample.readthedocs.io) for reprojection.
- **SO2 RGB recipe**: [CIRA quick
  guide](https://rammb.cira.colostate.edu/training/visit/quick_guides/Quick_Guide_SO2_RGB.pdf),
  transcribed to AHI channels in `config/satpy/`.
- **True Colour Reproduction**: [JMA's TCR
  method](http://www.jma.go.jp/jma/jma-eng/satellite/introduction/TCR.html),
  available as the `true_color_reproduction` composite if you prefer JMA's own
  colour rendering to satpy's default `true_color`.

If you publish any of this, credit JMA for the observations and JAXA for the
archive.